In [1]:
import pandas as pd
import json

def json_to_dataframe(file_path):
    """
    Read a JSON file and convert it to a pandas DataFrame.
    
    Parameters:
    -----------
    file_path : str
        Path to the JSON file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame created from the JSON data
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    df = pd.DataFrame(data)
    return df

In [3]:
df = json_to_dataframe('../job_scrapper/jobs.json')


In [4]:
df.head()

,title,company,location,link,description,skills
0,ARCHITECTE IT APPLICATIF,rh pro plus,"Tunis, Tunisie",https://tanitjobs.com/job/1992186/architecte-i...,Le portail de l'emploi en Tunisie\nConnexion I...,"[bpmn, ci/cd, docker, java, kubernetes, micros..."
1,Développeur PHP Symfony Senior H/F,septeo,"Tunis, Tunisie",https://tanitjobs.com/job/1981529/d%C3%A9velop...,Le portail de l'emploi en Tunisie\nConnexion I...,"[agile, css, elasticsearch, git, javascript, l..."
2,Frontend Developer (Angular),basispoint,"Sousse, Tunisie",https://tanitjobs.com/job/1999115/frontend-dev...,Le portail de l'emploi en Tunisie\nConnexion I...,"[agile, angular, git, html, scrum]"
3,Technicien Informatique / Support IT,ste cobam,"Medenine, Tunisie",https://tanitjobs.com/job/1994243/technicien-i...,Le portail de l'emploi en Tunisie\nConnexion I...,"[linux, networking, tcp/ip, vpn, windows server]"
4,Data Engineer / Python Developer,basispoint,"Sousse, Tunisie",https://tanitjobs.com/job/1999047/data-enginee...,Le portail de l'emploi en Tunisie\nConnexion I...,"[agile, data analysis, python, scrum]"


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        46 non-null     str   
 1   company      46 non-null     str   
 2   location     41 non-null     str   
 3   link         46 non-null     str   
 4   description  46 non-null     str   
 5   skills       46 non-null     object
dtypes: object(1), str(5)
memory usage: 124.7+ KB


In [ ]:
import sys
sys.path.append('..')

# Import the functions from controller
from controller import receive_profile_from_webhook, ProfilePayload

# Step 1: Load Jobs Data (already loaded in df)
print("=" * 60)
print("JOBS DATA (from job_scrapper/jobs.json)")
print("=" * 60)
print(f"Total jobs: {len(df)}")
print(f"Job columns: {df.columns.tolist()}\n")
print(df.head())

# Step 2: Simulate receiving profile data from webhook
print("\n" + "=" * 60)
print("PROFILE DATA (from webhook)")
print("=" * 60)

# Sample profile payload - this represents data received from webhook
sample_profile_payload = ProfilePayload(data=[
    {
        "skill": "Python",
        "experience_years": 5,
        "proficiency": "Advanced"
    },
    {
        "skill": "JavaScript", 
        "experience_years": 3,
        "proficiency": "Intermediate"
    },
    {
        "skill": "Data Analysis",
        "experience_years": 4,
        "proficiency": "Advanced"
    }
])

# Convert profile payload to DataFrame
profile_df = receive_profile_from_webhook(sample_profile_payload)
print(f"\nProfile Skills: {len(profile_df)}")
print(f"Profile columns: {profile_df.columns.tolist()}\n")
print(profile_df)

In [ ]:
# Summary: Both datasets ready for recommendation system
print("=" * 60)
print("SUMMARY - READY FOR JOB RECOMMENDATION")
print("=" * 60)

print(f"\nJOBS DATASET:")
print(f"   • Total jobs: {len(df)}")
print(f"   • Features: {df.columns.tolist()}")

print(f"\n👤 USER PROFILE DATASET:")
print(f"   • Skills: {len(profile_df)}")
print(f"   • Features: {profile_df.columns.tolist()}")

print(f"\n✓ Both datasets loaded successfully!")
print(f"✓ Ready to build recommendation system")

# Display sample data side by side
print("\n" + "=" * 60)
print("SAMPLE: First job vs User profile")
print("=" * 60)
print(f"\nFirst Job:\n{df.iloc[0]}")
print(f"\nUser's First Skill:\n{profile_df.iloc[0]}")

In [ ]:
# ============================================
# DATA PREPROCESSING & NORMALIZATION
# ============================================

import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import re
from difflib import SequenceMatcher

def preprocess_skills(skills_list):
    """
    Normalize skill names for consistent matching.
    
    Parameters:
    -----------
    skills_list : list
        List of skill strings
        
    Returns:
    --------
    list
        Normalized skills
    """
    if not isinstance(skills_list, list):
        return []
    
    normalized = []
    for skill in skills_list:
        # Convert to lowercase, remove extra spaces
        skill = str(skill).lower().strip()
        # Remove special characters but keep alphanumeric and hyphens/dots
        skill = re.sub(r'[^a-z0-9\-\.]', '', skill)
        if skill:
            normalized.append(skill)
    
    return list(set(normalized))  # Remove duplicates

def preprocess_location(location):
    """
    Normalize location names.
    
    Parameters:
    -----------
    location : str
        Location string
        
    Returns:
    --------
    str
        Normalized location
    """
    if pd.isna(location):
        return "unknown"
    
    location = str(location).lower().strip()
    # Extract main city (before comma)
    city = location.split(',')[0].strip()
    return city

# Apply preprocessing to jobs DataFrame
print("\n" + "=" * 60)
print("PREPROCESSING JOB DATA")
print("=" * 60)

df['skills_normalized'] = df['skills'].apply(preprocess_skills)
df['location_normalized'] = df['location'].apply(preprocess_location)

# Remove duplicates and fill missing values
df = df.drop_duplicates(subset=['title', 'company'], keep='first').reset_index(drop=True)
df['skills'] = df['skills'].fillna('').apply(lambda x: [] if x == '' else x)

print(f"\n✓ Preprocessed {len(df)} jobs")
print(f"✓ Sample normalized job:")
print(f"  Title: {df.iloc[0]['title']}")
print(f"  Skills: {df.iloc[0]['skills_normalized'][:5]}")
print(f"  Location: {df.iloc[0]['location_normalized']}")


In [ ]:
# Preprocess profile data
print("\n" + "=" * 60)
print("PREPROCESSING PROFILE DATA")
print("=" * 60)

# Normalize user skills
profile_df['skill_normalized'] = profile_df['skill'].apply(lambda x: preprocess_skills([str(x)]))
profile_df['skill_normalized'] = profile_df['skill_normalized'].apply(lambda x: x[0] if x else "")

user_skills = set(profile_df['skill_normalized'].tolist())
user_experience = profile_df['experience_years'].sum() if 'experience_years' in profile_df.columns else 0
user_proficiency = profile_df['proficiency'].tolist() if 'proficiency' in profile_df.columns else []

print(f"\n✓ User Profile Processed:")
print(f"  • Skills: {user_skills}")
print(f"  • Total Experience: {user_experience} years")
print(f"  • Proficiency Levels: {user_proficiency}")


In [ ]:
# ============================================
# JOB RECOMMENDATION ALGORITHM
# ============================================

def calculate_skill_match_score(user_skills, job_skills, weight_exact=0.8, weight_partial=0.4):
    """
    Calculate skill match score between user and job.
    
    Parameters:
    -----------
    user_skills : set
        Set of user skills
    job_skills : list
        List of job skills
    weight_exact : float
        Weight for exact matches (0-1)
    weight_partial : float
        Weight for partial/similar matches
        
    Returns:
    --------
    float
        Match score (0-1)
    """
    if not job_skills or not user_skills:
        return 0.0
    
    job_skills = preprocess_skills(job_skills)
    job_skills_set = set(job_skills)
    
    # Exact matches
    exact_matches = len(user_skills & job_skills_set)
    
    # Partial/similar matches (skills with sequence similarity > 0.6)
    partial_matches = 0
    for user_skill in user_skills:
        for job_skill in job_skills_set:
            if user_skill not in job_skill and job_skill not in user_skill:
                similarity = SequenceMatcher(None, user_skill, job_skill).ratio()
                if similarity > 0.6:
                    partial_matches += 1
                    break
    
    # Calculate score (normalize by job skills count)
    total_score = (exact_matches * weight_exact + partial_matches * weight_partial)
    max_possible = len(job_skills_set)
    
    return min(total_score / max_possible, 1.0) if max_possible > 0 else 0.0

def recommend_jobs(user_profile_df, jobs_df, top_n=5, preferred_locations=None):
    """
    Recommend jobs based on user profile.
    
    Parameters:
    -----------
    user_profile_df : pd.DataFrame
        User profile with skills
    jobs_df : pd.DataFrame
        Available jobs
    top_n : int
        Number of top recommendations
    preferred_locations : list
        User's preferred locations
        
    Returns:
    --------
    pd.DataFrame
        Top N recommended jobs with scores
    """
    user_skills = set(user_profile_df['skill_normalized'].tolist())
    
    # Calculate scores for each job
    recommendations = []
    
    for idx, job in jobs_df.iterrows():
        job_skills = job['skills'] if isinstance(job['skills'], list) else []
        
        # Skill match score (60% weight)
        skill_score = calculate_skill_match_score(user_skills, job_skills)
        
        # Location score (20% weight) - boost score if matches preferred location
        location_score = 0.5  # baseline
        if preferred_locations and job['location_normalized'] in preferred_locations:
            location_score = 1.0
        
        # Skills requirement score based on match ratio (20% weight)
        matched_skills = len(set(preprocess_skills(job_skills)) & user_skills)
        total_required = len(preprocess_skills(job_skills)) if job_skills else 1
        requirement_score = matched_skills / total_required
        
        # Weighted final score
        final_score = (skill_score * 0.6) + (location_score * 0.2) + (requirement_score * 0.2)
        
        recommendations.append({
            'job_title': job['title'],
            'company': job['company'],
            'location': job['location'],
            'skills_needed': job_skills,
            'matched_skills': list(set(preprocess_skills(job_skills)) & user_skills),
            'skill_match_score': round(skill_score, 3),
            'location_score': round(location_score, 3),
            'final_score': round(final_score, 3),
            'link': job['link']
        })
    
    # Sort by final score
    recommendations_df = pd.DataFrame(recommendations).sort_values('final_score', ascending=False)
    
    return recommendations_df.head(top_n)

# Generate recommendations
print("\n" + "=" * 60)
print("GENERATING JOB RECOMMENDATIONS")
print("=" * 60)

# Get preferred locations (or use all)
preferred_locations = None  # Change to ['tunis', 'sousse'] if needed

recommendations = recommend_jobs(
    user_profile_df=profile_df,
    jobs_df=df,
    top_n=5,
    preferred_locations=preferred_locations
)

print(f"\n✓ Generated {len(recommendations)} recommendations\n")
print(recommendations[['job_title', 'company', 'location', 'final_score', 'matched_skills']].to_string(index=False))


In [ ]:
# ============================================
# DETAILED RECOMMENDATIONS VIEW
# ============================================

print("\n" + "=" * 80)
print("TOP RECOMMENDED JOBS FOR YOUR PROFILE")
print("=" * 80)

for idx, (_, job) in enumerate(recommendations.iterrows(), 1):
    print(f"\n{'─' * 80}")
    print(f"#{idx} - {job['job_title']} (Score: {job['final_score']:.1%})")
    print(f"{'─' * 80}")
    print(f"Company:        {job['company']}")
    print(f"Location:       {job['location']}")
    print(f"Link:           {job['link']}")
    print(f"\nSkills Analysis:")
    print(f"  • Your Skills Found:  {', '.join(job['matched_skills']) if job['matched_skills'] else 'None'}")
    print(f"  • Skills Match Score: {job['skill_match_score']:.1%}")
    print(f"  • Location Match:     {job['location_score']:.1%}")
    
    all_required = set(preprocess_skills(job['skills_needed']))
    missing_skills = all_required - set(job['matched_skills'])
    if missing_skills:
        print(f"  • Skills to Learn:    {', '.join(list(missing_skills)[:5])}")
    print(f"  • Overall Score:      {job['final_score']:.1%}")

# Summary statistics
print(f"\n{'=' * 80}")
print("RECOMMENDATION SUMMARY")
print(f"{'=' * 80}")
print(f"Average Recommendation Score: {recommendations['final_score'].mean():.1%}")
print(f"Best Match Score:             {recommendations['final_score'].max():.1%}")
print(f"Total Skills in Profile:      {len(user_skills)}")
print(f"Average Matched Skills/Job:   {recommendations['matched_skills'].apply(len).mean():.1f}")
print(f"\n✓ Recommendations ready! Apply for jobs with highest scores.")


In [ ]:
# ============================================
# SEND RECOMMENDATIONS TO CONTROLLER
# ============================================

from controller import send_recommendations_to_controller

# Prepare recommendations for export
recommendations_data = recommendations.copy()
recommendations_data['matched_skills'] = recommendations_data['matched_skills'].apply(lambda x: ', '.join(x))
recommendations_data['skills_needed'] = recommendations_data['skills_needed'].apply(lambda x: ', '.join(preprocess_skills(x)))

# Convert to dict format for transmission
recommendations_list = recommendations_data.to_dict(orient='records')

print("\n" + "=" * 60)
print("EXPORTING RECOMMENDATIONS TO CONTROLLER")
print("=" * 60)

# Send to controller
result = send_recommendations_to_controller(recommendations_list)

print(f"\n✓ Status: {result['status']}")
print(f"✓ Recommendations exported: {result['count']}")
print(f"✓ Access via webhook: POST /webhook/get-recommendations")
print(f"\nReady to send to client via webhook!")


In [ ]:
# ============================================
# RETRIEVE RECOMMENDATIONS FROM CONTROLLER
# ============================================

# Display endpoint information
print("\n" + "=" * 60)
print("WEBHOOK ENDPOINTS - RECOMMENDATIONS")
print("=" * 60)

print(f"""
To retrieve recommendations from controller:

1. GET RECOMMENDATIONS:
   GET http://localhost:8000/webhook/get-recommendations
   
   Returns:
   {{
       "status": "success",
       "recommendations": [
           {{
               "job_title": "...",
               "company": "...",
               "location": "...",
               "final_score": 0.85,
               "matched_skills": ["Python", "JavaScript"],
               ...
           }},
           ...
       ],
       "count": 5
   }}

2. SEND RECOMMENDATIONS:
   POST http://localhost:8000/webhook/send-recommendations
   
   Optional query params:
   - client_url: URL to forward recommendations to

3. WORKFLOW:
   Notebook (generate) → Controller (store) → Webhook (retrieve) → Client
""")

print(f"✓ Recommendations are now available on the webhook!")
print(f"✓ Total recommendations stored: {len(recommendations_list)}")
print(f"✓ Status: Ready to be consumed by clients")
